# Automated Customer Reviews — Classification, Clustering & Generative AI Summarization

**Project | Business Case: Automated Customer Reviews**

This notebook implements the 3 main tasks :

***Sentiment CLASSIFICATION** — classify each review as *positive / negative / neutral* based on the star rating, then with a pretrained Transformer model.

***Product category CLUSTERING** — group products into 4 to 6 meta-categories.

***Generative AI SUMMARIZATION** — generate a short article (blog-post style) per meta-category, with the top 3 products, their main complaints, and the worst product to avoid.

The notebook ends with an interactive demo (deployment bonus) built with #############.

> **Team:** Lydia, Marcelo, Don.

> **Date:** 15_06_2026

**ROADMAP**

1. DATASET ACQUISITION

2. DATA UNDERSTANDING

3. CLEANING
4. LABEL CREATION
5. EDA
6. BALANCING
7. BASELINE MODEL : Classification sentiments with pretrained transformer model on HF.
8. COMPARISON MODEL
9. EVALUATION
10. CLUSTERING : create categories products with KMeans
11. SUMMARIZATION : generate short articles summarization with generative AI
12. APP
13. REPORT + DERIVABLES


**1.DATASET ACQUISITION**

1.1 Import libraries

1.2 Data loadind

In [1]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn transformers datasets evaluate accelerate sentence-transformers gradio torch kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [2]:
import os
import re
import glob
import json
import gzip
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, silhouette_score
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

from sentence_transformers import SentenceTransformer
import gradio as gr

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


1.3 Project Configuration

This section defines all project paths and execution options.

In [3]:
# ============================================================
# Paths
# ============================================================
DATA_DIR = "data"
PRIMARY_DIR = os.path.join(DATA_DIR, "primary")
LARGER_DIR = os.path.join(DATA_DIR, "larger")
OUTPUT_DIR = "outputs"

os.makedirs(PRIMARY_DIR, exist_ok=True)
os.makedirs(LARGER_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# Dataset limits
# ============================================================
MAX_PRIMARY_ROWS = None          # None = use all available primary rows
MAX_LARGER_ROWS = 100000          # UCSD is huge. Start small.
MAX_BALANCED_PER_CLASS = 10000    # Increase if Colab GPU can handle it
MAX_CLUSTER_ROWS = 5000

# ============================================================
# Model configuration
# ============================================================
BASELINE_MODEL_NAME = "distilbert-base-uncased"
COMPARISON_MODEL_NAME = "roberta-base"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SUMMARIZER_MODEL_NAME = "facebook/bart-large-cnn"

RUN_TRAINING = True
RUN_COMPARISON_MODEL = True
RUN_SPECIALIST_MODELS = False    # Set True only if you have enough time/GPU
RUN_GRADIO_APP = True

# Training parameters
NUM_EPOCHS = 2
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
MAX_TOKEN_LENGTH = 256

**2. DATA UNDERSTANDING**

The project requires two datasets.

### Primary dataset — Kaggle

The primary dataset is available on Kaggle. Since Kaggle requires authentication, and we have to download it.

### Larger dataset — UCSD Amazon Reviews



In [4]:
# ============================================================
# PRIMARY DATASET
# ============================================================

from google.colab import files
files.upload()  # Upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d datafiniti/consumer-reviews-of-amazon-products -p data/primary --unzip

# After download, the primary CSV should appear inside data/primary/

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/datafiniti/consumer-reviews-of-amazon-products
License(s): CC-BY-NC-SA-4.0
100% 16.3M/16.3M [00:00<00:00, 112MB/s] 



In [5]:
import pandas as pd

df_primary = pd.read_csv("data/primary/1429_1.csv")

print(df_primary.shape)
df_primary.head(10)

(34660, 21)


,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42
5,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This amazon fire 8 inch tablet is the perfect ...,Just what we expected,NaN,NaN,tklit
6,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,"Great for e-reading on the go, nice and light ...",great e-reader tablet,NaN,NaN,Droi
7,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,"I gave this as a Christmas gift to my inlaws, ...",Great for gifts,NaN,NaN,Kacy
8,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",

In [6]:
df_primary.columns

Index(['id', 'name', 'asins', 'brand', 'categories', 'keys', 'manufacturer',
       'reviews.date', 'reviews.dateAdded', 'reviews.dateSeen',
       'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id',
       'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs',
       'reviews.text', 'reviews.title', 'reviews.userCity',
       'reviews.userProvince', 'reviews.username'],
      dtype='object')

In [7]:
############______________________################

# LARGER DATASET

############______________________################


import os
import json
import gzip
import requests
import pandas as pd
from tqdm import tqdm

LARGER_DIR = "data/larger"
os.makedirs(LARGER_DIR, exist_ok=True)

SAMPLE_PER_CATEGORY = 10000

BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories"

CATEGORIES = [
    "All_Beauty",
    "Amazon_Fashion",
    "Appliances",
    "Arts_Crafts_and_Sewing",
    "Automotive",
    "Baby_Products",
    "Beauty_and_Personal_Care",
    "Books",
    "CDs_and_Vinyl",
    "Cell_Phones_and_Accessories",
    "Clothing_Shoes_and_Jewelry",
    "Digital_Music",
    "Electronics",
    "Gift_Cards",
    "Grocery_and_Gourmet_Food",
    "Handmade_Products",
    "Health_and_Household",
    "Health_and_Personal_Care",
    "Home_and_Kitchen",
    "Industrial_and_Scientific",
    "Kindle_Store",
    "Magazine_Subscriptions",
    "Movies_and_TV",
    "Musical_Instruments",
    "Office_Products",
    "Patio_Lawn_and_Garden",
    "Pet_Supplies",
    "Software",
    "Sports_and_Outdoors",
    "Subscription_Boxes",
    "Tools_and_Home_Improvement",
    "Toys_and_Games",
    "Video_Games"
]

def map_rating_to_sentiment(rating):
    rating = float(rating)
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

def stream_category_sample(category, sample_size=10000):
    url = f"{BASE_URL}/{category}.jsonl.gz"
    rows = []

    print(f"\nStreaming category: {category}")
    print(url)

    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()

        with gzip.GzipFile(fileobj=response.raw) as gz:
            for line in tqdm(gz, desc=category):
                if len(rows) >= sample_size:
                    break

                try:
                    item = json.loads(line.decode("utf-8"))
                except Exception:
                    continue

                text = item.get("text", "")
                title = item.get("title", "")
                rating = item.get("rating", None)

                if not text or rating is None:
                    continue

                review_text = f"{title}. {text}" if title else text

                rows.append({
                    "review_text": review_text,
                    "rating": float(rating),
                    "sentiment": map_rating_to_sentiment(rating),
                    "product_name": item.get("parent_asin", item.get("asin", "Unknown Product")),
                    "raw_category": category,
                    "source": "amazon_reviews_2023"
                })

    except Exception as e:
        print(f"[ERROR] Failed category {category}: {e}")

    return pd.DataFrame(rows)

all_dfs = []

for category in CATEGORIES:
    category_df = stream_category_sample(category, SAMPLE_PER_CATEGORY)

    print(f"{category}: {category_df.shape[0]} reviews collected")

    if not category_df.empty:
        all_dfs.append(category_df)

larger_df = pd.concat(all_dfs, ignore_index=True)

larger_df = larger_df.dropna(subset=["review_text", "rating", "sentiment"])
larger_df = larger_df.drop_duplicates(subset=["review_text"])

output_path = f"{LARGER_DIR}/amazon_reviews_2023_all_categories_sampled.csv"
larger_df.to_csv(output_path, index=False)

print("\nFINAL SHAPE:", larger_df.shape)
print("Saved to:", output_path)

display(larger_df.head())
display(larger_df["raw_category"].value_counts())
display(larger_df["sentiment"].value_counts())


Streaming category: All_Beauty
https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz


All_Beauty: 10000it [00:00, 66121.90it/s]


All_Beauty: 10000 reviews collected

Streaming category: Amazon_Fashion
https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Amazon_Fashion.jsonl.gz


Amazon_Fashion: 10000it [00:00, 72767.62it/s]


Amazon_Fashion: 10000 reviews collected

Streaming category: Appliances
https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Appliances.jsonl.gz


...

Video_Games: 10000it [00:00, 59245.68it/s]


Video_Games: 10000 reviews collected

FINAL SHAPE: (317201, 6)
Saved to: data/larger/amazon_reviews_2023_all_categories_sampled.csv


,review_text,rating,sentiment,product_name,raw_category,source
0,Such a lovely scent but not overpowering.. Thi...,5.0,positive,B00YQ6X8EO,All_Beauty,amazon_reviews_2023
1,Works great but smells a little weird.. This p...,4.0,positive,B081TJ8YS3,All_Beauty,amazon_reviews_2023
2,"Yes!. Smells good, feels great!",5.0,positive,B097R46CSY,All_Beauty,amazon_reviews_2023
3,Synthetic feeling. Felt synthetic,1.0,negative,B09JS339BZ,All_Beauty,amazon_reviews_2023
4,A+. Love it,5.0,positive,B08BZ63GMJ,All_Beauty,amazon_reviews_2023


,count
raw_category,
All_Beauty,9905
Subscription_Boxes,9876
Kindle_Store,9860
Amazon_Fashion,9859
Beauty_and_Personal_Care,9830
Appliances,9781
Books,9777
Handmade_Products,9749
Musical_Instruments,9746


,count
sentiment,
positive,258775
negative,33851
neutral,24575


In [8]:
larger_df = pd.read_csv("data/larger/amazon_reviews_2023_all_categories_sampled.csv")

print(larger_df.shape)
larger_df.head()

(317201, 6)


,review_text,rating,sentiment,product_name,raw_category,source
0,Such a lovely scent but not overpowering.. Thi...,5.0,positive,B00YQ6X8EO,All_Beauty,amazon_reviews_2023
1,Works great but smells a little weird.. This p...,4.0,positive,B081TJ8YS3,All_Beauty,amazon_reviews_2023
2,"Yes!. Smells good, feels great!",5.0,positive,B097R46CSY,All_Beauty,amazon_reviews_2023
3,Synthetic feeling. Felt synthetic,1.0,negative,B09JS339BZ,All_Beauty,amazon_reviews_2023
4,A+. Love it,5.0,positive,B08BZ63GMJ,All_Beauty,amazon_reviews_2023


**3. CLEANING**

In this section we define some helper and reusable functions for:

- Cleaning review text.
- Mapping star ratings to sentiment labels.
- Loading the primary Kaggle dataset.
- Loading the larger UCSD dataset.
- Balancing the dataset.
- Training Hugging Face models.
- Plotting evaluation results.

The rating-to-sentiment mapping follows the project brief:

| Star rating | Sentiment |
|---|---|
| 1–2 | Negative |
| 3 | Neutral |
| 4–5 | Positive |

In [9]:
def clean_text(text):
    """Basic text cleaning for review text. It preserves normal word spacing."""
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r'[^A-Za-z0-9À-ÿ.,!?;:"()€$%&/\-\s]', " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def map_rating_to_sentiment(rating):
    """Maps 1-5 star ratings into the 3 required sentiment classes."""
    try:
        rating = float(rating)
    except Exception:
        return np.nan

    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    elif rating >= 4:
        return "positive"
    return np.nan


label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}


def normalize_product_name(x):
    """
    Cleans product names while preserving readable spacing.

    Important:
    - Kaggle/Datafiniti often gives real product names.
    - Amazon Reviews 2023 review files often give ASINs instead of real names.
      We do not pretend those ASINs are product names.
    """
    if pd.isna(x):
        return "Unknown Product"

    x = str(x)
    x = x.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    x = re.sub(r",+", ",", x)
    x = re.sub(r"\s*,\s*", ", ", x)      # comma followed by a space
    x = re.sub(r"\s*-\s*", " - ", x)      # readable hyphen separation
    x = re.sub(r"\s+", " ", x)
    x = x.strip(" ,-")

    if len(x) == 0 or x.lower() in {"nan", "none", "null"}:
        return "Unknown Product"
    return x


def is_asin(x):
    """Detects Amazon ASIN-like product identifiers."""
    return bool(re.fullmatch(r"B[0-9A-Z]{9}", str(x).strip()))


def product_id_type(x):
    """Classifies a product identifier as named product, ASIN, or unknown."""
    x = normalize_product_name(x)
    if x == "Unknown Product":
        return "unknown"
    if is_asin(x):
        return "asin"
    return "named_product"


def normalize_category(x):
    """
    Normalizes category values.

    - Kaggle/Datafiniti often has comma-separated category paths.
      We keep only the first level, e.g. 'Electronics,iPad & Tablets' -> 'Electronics'.
    - Amazon Reviews 2023 already has one category name, e.g. 'Books'.
    """
    if pd.isna(x):
        return "Unknown Category"

    x = str(x)
    x = x.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    x = x.replace("[", "").replace("]", "")
    x = x.replace("{", "").replace("}", "")
    x = x.replace("'", "").replace('"', "")
    x = re.sub(r"\s+", " ", x).strip()

    if "," in x:
        x = x.split(",")[0].strip()

    if len(x) == 0 or x.lower() in {"nan", "none", "null"}:
        return "Unknown Category"
    return x


def make_product_display_name(product_name, raw_category):
    """
    Creates a safe display name for EDA and reports.

    Real names are preserved.
    ASIN-only products are shown as category products with the ASIN in parentheses.
    Unknown products remain explicit and can be excluded from product-level charts.
    """
    product_name = normalize_product_name(product_name)
    category = normalize_category(raw_category)

    if product_name == "Unknown Product":
        return "Unknown Product"
    if is_asin(product_name):
        return f"{category} product ({product_name})"
    return product_name


def shorten_label(x, max_len=70):
    """Shortens very long labels for readable plots/tables."""
    x = str(x)
    return x if len(x) <= max_len else x[:max_len - 3] + "..."


def find_first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def stratified_limit_by_category(dataframe, max_rows, category_col="raw_category"):
    """
    Limits a large dataset while preserving category diversity.

    Do not use plain .head(max_rows) here: the Amazon Reviews 2023 sampled CSV may be
    ordered by category, so .head() can silently keep only the first few categories.
    """
    if max_rows is None or len(dataframe) <= max_rows:
        return dataframe.copy().reset_index(drop=True)

    if category_col not in dataframe.columns:
        return dataframe.sample(max_rows, random_state=RANDOM_STATE).reset_index(drop=True)

    n_categories = dataframe[category_col].nunique()
    per_category = max(1, int(np.ceil(max_rows / n_categories)))

    sampled = (
        dataframe
        .groupby(category_col, group_keys=False)
        .apply(lambda x: x.sample(min(len(x), per_category), random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )

    if len(sampled) > max_rows:
        sampled = sampled.sample(max_rows, random_state=RANDOM_STATE).reset_index(drop=True)

    return sampled

In [10]:
def load_primary_dataset(primary_dir):
    """Loads the Kaggle/Datafiniti primary dataset and converts it to the unified schema."""
    files = []
    files.extend(glob.glob(os.path.join(primary_dir, "*.csv")))
    files.extend(glob.glob(os.path.join(primary_dir, "*.tsv")))

    if len(files) == 0:
        print(f"[WARNING] No primary dataset file found in {primary_dir}")
        return pd.DataFrame()

    parts = []

    for file in files:
        print("Loading primary file:", file)

        if file.endswith(".tsv"):
            temp = pd.read_csv(file, sep="\t", low_memory=False)
        else:
            temp = pd.read_csv(file, low_memory=False)

        if MAX_PRIMARY_ROWS is not None:
            temp = temp.sample(min(MAX_PRIMARY_ROWS, len(temp)), random_state=RANDOM_STATE)

        text_col = find_first_existing_column(temp, [
            "reviews.text", "review_text", "text", "reviewText", "reviews"
        ])
        rating_col = find_first_existing_column(temp, [
            "reviews.rating", "rating", "overall", "stars", "reviews.stars"
        ])
        product_col = find_first_existing_column(temp, [
            "name", "product_name", "title", "asins", "asin"
        ])
        category_col = find_first_existing_column(temp, [
            "categories", "primaryCategories", "category", "main_cat"
        ])

        if text_col is None or rating_col is None:
            print("[WARNING] Skipping file because text/rating columns were not found:", file)
            print("Available columns:", temp.columns.tolist())
            continue

        part = pd.DataFrame()
        part["review_text"] = temp[text_col].apply(clean_text)
        part["rating"] = pd.to_numeric(temp[rating_col], errors="coerce")
        part["product_name"] = temp[product_col].apply(normalize_product_name) if product_col else "Unknown Product"
        part["raw_category"] = temp[category_col].apply(normalize_category) if category_col else "Unknown Category"
        part["source"] = "primary"
        parts.append(part)

    if len(parts) == 0:
        return pd.DataFrame()

    return pd.concat(parts, ignore_index=True)


def load_larger_ucsd_dataset(folder, max_rows=None):
    """
    Loads the larger Amazon Reviews 2023 / UCSD-style dataset.

    Supported files in data/larger/:
    - sampled CSV created by the helper cell
    - .jsonl.gz / .json.gz / .json line files

    Important fix:
    For CSV files, we do stratified sampling by category instead of .head(max_rows),
    otherwise the first categories in the file dominate the larger dataset.
    """
    frames = []

    if not os.path.exists(folder):
        print(f"[WARNING] Larger dataset folder not found: {folder}")
        return pd.DataFrame()

    csv_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".csv")]

    for path in csv_files:
        print(f"Loading larger CSV file: {path}")
        temp = pd.read_csv(path, low_memory=False)

        required_cols = ["review_text", "rating", "product_name", "raw_category", "source"]
        if not all(col in temp.columns for col in required_cols):
            raise ValueError(f"CSV file {path} does not have the expected schema: {required_cols}")

        temp = temp[required_cols].copy()
        temp["review_text"] = temp["review_text"].apply(clean_text)
        temp["rating"] = pd.to_numeric(temp["rating"], errors="coerce")
        temp["product_name"] = temp["product_name"].apply(normalize_product_name)
        temp["raw_category"] = temp["raw_category"].apply(normalize_category)
        temp["source"] = temp["source"].fillna("amazon_reviews_2023")

        temp = stratified_limit_by_category(temp, max_rows=max_rows, category_col="raw_category")
        frames.append(temp)

    json_files = [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.endswith(".json") or f.endswith(".json.gz") or f.endswith(".jsonl.gz")
    ]

    for path in json_files:
        print(f"Loading larger JSON file: {path}")
        rows = []
        opener = gzip.open if path.endswith(".gz") else open
        category_from_file = os.path.basename(path).replace(".jsonl.gz", "").replace(".json.gz", "").replace(".json", "")

        with opener(path, "rt", encoding="utf-8") as f:
            for line in f:
                if max_rows is not None and len(rows) >= max_rows:
                    break
                try:
                    item = json.loads(line)
                except Exception:
                    continue

                text = item.get("text", item.get("reviewText", ""))
                title = item.get("title", item.get("summary", ""))
                rating = item.get("rating", item.get("overall", None))

                if not text or rating is None:
                    continue

                review_text = f"{title}. {text}" if title else text
                rows.append({
                    "review_text": clean_text(review_text),
                    "rating": float(rating),
                    "product_name": normalize_product_name(item.get("parent_asin", item.get("asin", "Unknown Product"))),
                    "raw_category": normalize_category(category_from_file),
                    "source": "amazon_reviews_2023"
                })

        if rows:
            temp = pd.DataFrame(rows)
            temp = stratified_limit_by_category(temp, max_rows=max_rows, category_col="raw_category")
            frames.append(temp)

    if not frames:
        print(f"[WARNING] No larger dataset file found in {folder}")
        return pd.DataFrame()

    larger_df = pd.concat(frames, ignore_index=True)
    print("Loaded larger dataset shape:", larger_df.shape)
    return larger_df

In [11]:
def balance_dataset(df, max_per_class=3000):
    """
    Downsamples each sentiment class to the same maximum size.

    This is necessary because Amazon reviews are usually highly positive-heavy.
    Without balancing, a model can achieve misleading accuracy by mostly predicting positive.
    """
    balanced_parts = []

    for sentiment in ["negative", "neutral", "positive"]:
        part = df[df["sentiment"] == sentiment]
        n = min(len(part), max_per_class)

        if n == 0:
            print(f"[WARNING] No rows for class: {sentiment}")
            continue

        balanced_parts.append(part.sample(n, random_state=RANDOM_STATE))

    balanced = pd.concat(balanced_parts, ignore_index=True)
    balanced = balanced.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return balanced


def plot_count(series, title, xlabel):
    plt.figure(figsize=(8, 4))
    order = series.value_counts().index
    sns.countplot(x=series, order=order)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.xticks(rotation=30)
    plt.show()


def plot_confusion_matrix(y_true, y_pred, title):
    labels = [0, 1, 2]
    label_names = ["negative", "neutral", "positive"]
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=label_names,
        yticklabels=label_names
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

In [12]:
def tokenize_for_transformer(train_df, test_df, model_name, max_length=256):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_dataset = Dataset.from_pandas(train_df[["review_text", "label"]])
    test_dataset = Dataset.from_pandas(test_df[["review_text", "label"]])

    def tokenize(batch):
        return tokenizer(
            batch["review_text"],
            padding="max_length",
            truncation=True,
            max_length=max_length
        )

    train_dataset = train_dataset.map(tokenize, batched=True)
    test_dataset = test_dataset.map(tokenize, batched=True)

    train_dataset = train_dataset.remove_columns(["review_text"])
    test_dataset = test_dataset.remove_columns(["review_text"])

    train_dataset.set_format("torch")
    test_dataset.set_format("torch")

    return tokenizer, train_dataset, test_dataset


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }


def train_transformer_classifier(
    train_df,
    test_df,
    model_name,
    output_subdir,
    epochs=2,
    batch_size=8,
    learning_rate=2e-5
):
    """
    Fine-tunes a Hugging Face Transformer model for 3-class sentiment classification.
    """
    print("\nTraining model:", model_name)

    tokenizer, train_dataset, test_dataset = tokenize_for_transformer(
        train_df,
        test_df,
        model_name=model_name,
        max_length=MAX_TOKEN_LENGTH
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id
    )

    output_path = os.path.join(OUTPUT_DIR, output_subdir)

    args = TrainingArguments(
        output_dir=output_path,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = test_df["label"].values

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))

    plot_confusion_matrix(y_true, y_pred, f"Confusion Matrix - {model_name}")

    trainer.save_model(output_path)
    tokenizer.save_pretrained(output_path)

    return {
        "model_name": model_name,
        "output_path": output_path,
        "trainer": trainer,
        "tokenizer": tokenizer,
        "y_true": y_true,
        "y_pred": y_pred,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted")
    }

In [13]:
def train_specialist_binary_model(
    train_df,
    test_df,
    target_sentiment,
    base_model_name="distilbert-base-uncased",
    epochs=1,
    batch_size=8
):
    """
    Trains a one-vs-rest specialist model.

    Examples:
    - positive specialist: positive vs not_positive
    - negative specialist: negative vs not_negative
    - neutral specialist: neutral vs not_neutral

    This is optional and computationally heavier.
    """
    print(f"\nTraining specialist model for: {target_sentiment}")

    temp_train = train_df.copy()
    temp_test = test_df.copy()

    temp_train["label"] = (temp_train["sentiment"] == target_sentiment).astype(int)
    temp_test["label"] = (temp_test["sentiment"] == target_sentiment).astype(int)

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    train_dataset = Dataset.from_pandas(temp_train[["review_text", "label"]])
    test_dataset = Dataset.from_pandas(temp_test[["review_text", "label"]])

    def tokenize(batch):
        return tokenizer(
            batch["review_text"],
            padding="max_length",
            truncation=True,
            max_length=MAX_TOKEN_LENGTH
        )

    train_dataset = train_dataset.map(tokenize, batched=True)
    test_dataset = test_dataset.map(tokenize, batched=True)

    train_dataset = train_dataset.remove_columns(["review_text"])
    test_dataset = test_dataset.remove_columns(["review_text"])

    train_dataset.set_format("torch")
    test_dataset.set_format("torch")

    model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        num_labels=2,
        id2label={0: f"not_{target_sentiment}", 1: target_sentiment},
        label2id={f"not_{target_sentiment}": 0, target_sentiment: 1}
    )

    output_path = os.path.join(OUTPUT_DIR, f"specialist_{target_sentiment}")

    def binary_compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "weighted_f1": f1_score(labels, preds, average="weighted")
        }

    args = TrainingArguments(
        output_dir=output_path,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="weighted_f1",
        greater_is_better=True,
        report_to="none",
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        tokenizer=tokenizer,
        compute_metrics=binary_compute_metrics
    )

    trainer.train()
    trainer.save_model(output_path)
    tokenizer.save_pretrained(output_path)

    return {
        "sentiment": target_sentiment,
        "output_path": output_path,
        "trainer": trainer,
        "tokenizer": tokenizer
    }

**4. LOAD AND MERGE THE TWO DATASETS**

This section loads:

1. The primary Kaggle/Datafiniti dataset.
2. The larger UCSD Amazon Reviews dataset.

Both datasets are merged into a unified schema, that is required before performing EDA and training the model.

In [15]:
# ============================================================
# Load Primary + Larger Dataset
# ============================================================

primary_df = load_primary_dataset(PRIMARY_DIR)
larger_df = load_larger_ucsd_dataset(LARGER_DIR, max_rows=None)

print("Primary dataset shape:", primary_df.shape)
print("Larger dataset shape:", larger_df.shape)

if primary_df.empty:
    raise ValueError("Primary dataset was not loaded.")

if larger_df.empty:
    raise ValueError("Larger dataset was not loaded.")

# Create sentiment before filtering
primary_df["sentiment"] = primary_df["rating"].apply(map_rating_to_sentiment)
larger_df["sentiment"] = larger_df["rating"].apply(map_rating_to_sentiment)

primary_df = primary_df.dropna(subset=["review_text", "rating", "sentiment"])
larger_df = larger_df.dropna(subset=["review_text", "rating", "sentiment"])

print("\nPrimary sentiment distribution:")
print(primary_df["sentiment"].value_counts())

print("\nLarger sentiment distribution before filtering:")
print(larger_df["sentiment"].value_counts())

# Keep only neutral and negative reviews from the larger dataset
larger_minorities_df = larger_df[
    larger_df["sentiment"].isin(["negative", "neutral"])
].copy()

print("\nLarger dataset after keeping only neutral and negative:")
print(larger_minorities_df["sentiment"].value_counts())

# Final merged dataset:
# full primary dataset + only neutral/negative reviews from larger dataset
df = pd.concat(
    [primary_df, larger_minorities_df],
    ignore_index=True
)

df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nMerged dataset shape:", df.shape)
print("\nMerged sentiment distribution:")
print(df["sentiment"].value_counts())

display(df.head())

Loading primary file: data/primary/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv
Loading primary file: data/primary/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv
Loading primary file: data/primary/1429_1.csv
Loading larger CSV file: data/larger/amazon_reviews_2023_all_categories_sampled.csv
Loaded larger dataset shape: (317201, 5)
Primary dataset shape: (67992, 5)
Larger dataset shape: (317201, 5)

Primary sentiment distribution:
sentiment
positive    62547
neutral      2902
negative     2510
Name: count, dtype: int64

Larger sentiment distribution before filtering:
sentiment
positive    258775
negative     33851
neutral      24575
Name: count, dtype: int64

Larger dataset after keeping only neutral and negative:
sentiment
negative    33851
neutral     24575
Name: count, dtype: int64

Merged dataset shape: (126385, 6)

Merged sentiment distribution:
sentiment
positive    62547
negative    36361
neutral     27477
Name: count, dtype: int64


,review_text,rating,product_name,raw_category,source,sentiment
0,Looks and feels cheap. Very thin leather and t...,2.0,B01HFL7CY8,Musical_Instruments,amazon_reviews_2023,negative
1,My daughter like the tablet and she use the ta...,4.0,"Fire Tablet, 7 Display, Wi - Fi, 16 GB - Inclu...",Fire Tablets,primary,positive
2,"Okay, but not a beauty box. I was so excited, ...",3.0,B07N1572VL,Subscription_Boxes,amazon_reviews_2023,neutral
3,"Looks nice fit well, low quality materials, ju...",2.0,B07FKP1HQM,Amazon_Fashion,amazon_reviews_2023,negative
4,This is a recycled book.. It was sold a year a...,1.0,B09F753DVV,Kindle_Store,amazon_reviews_2023,negative


LOADING DATASET FROM HUGGING FACE

In [16]:
from datasets import load_dataset

hf_ds = load_dataset("SetFit/amazon_reviews_multi_en")

print(hf_ds)
print(hf_ds["train"].features)

hf_train = hf_ds["train"].to_pandas()

display(hf_train.head())
print(hf_train["label"].value_counts().sort_index())

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/200000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
})
{'id': Value('string'), 'text': Value('string'), 'label': Value('int64'), 'label_text': Value('string')}


,id,text,label,label_text
0,en_0964290,Arrived broken. Manufacturer defect. Two of th...,0,0
1,en_0690095,the cabinet dot were all detached from backing...,0,0
2,en_0311558,I received my first order of this product and ...,0,0
3,en_0044972,This product is a piece of shit. Do not buy. D...,0,0
4,en_0784379,went through 3 in one day doesn't fit correct ...,0,0


label
0    40000
1    40000
2    40000
3    40000
4    40000
Name: count, dtype: int64


In [17]:
def hf_label_to_sentiment(label):
    if label in [0, 1]:
        return "negative"
    elif label == 2:
        return "neutral"
    elif label in [3, 4]:
        return "positive"
    else:
        return None

hf_train["sentiment"] = hf_train["label"].apply(hf_label_to_sentiment)

print(hf_train["sentiment"].value_counts())

sentiment
negative    80000
positive    80000
neutral     40000
Name: count, dtype: int64


In [28]:
hf_extra_df = hf_train[
    hf_train["sentiment"].isin(["negative", "neutral"])
].copy()

print(hf_extra_df["sentiment"].value_counts())
display(hf_extra_df.head())

sentiment
negative    80000
neutral     40000
Name: count, dtype: int64


,id,text,label,label_text,sentiment
0,en_0964290,Arrived broken. Manufacturer defect. Two of th...,0,0,negative
1,en_0690095,the cabinet dot were all detached from backing...,0,0,negative
2,en_0311558,I received my first order of this product and ...,0,0,negative
3,en_0044972,This product is a piece of shit. Do not buy. D...,0,0,negative
4,en_0784379,went through 3 in one day doesn't fit correct ...,0,0,negative


In [29]:
# Clean Hugging Face dataset before sampling

hf_clean = hf_train.copy()

# Rename text column to match project schema
hf_clean = hf_clean.rename(columns={
    "text": "review_text"
})

# Keep only useful columns first
hf_clean = hf_clean[
    [
        "review_text",
        "label",
        "sentiment"
    ]
].copy()

# Clean review text
hf_clean["review_text"] = hf_clean["review_text"].fillna("").astype(str)
hf_clean["review_text"] = hf_clean["review_text"].str.replace("\n", " ", regex=False)
hf_clean["review_text"] = hf_clean["review_text"].str.replace("\r", " ", regex=False)
hf_clean["review_text"] = hf_clean["review_text"].str.replace("\t", " ", regex=False)
hf_clean["review_text"] = hf_clean["review_text"].str.replace(r"\s+", " ", regex=True)
hf_clean["review_text"] = hf_clean["review_text"].str.strip()

# Remove empty or too short reviews
hf_clean = hf_clean[hf_clean["review_text"] != ""]
hf_clean = hf_clean[hf_clean["review_text"].str.len() >= 10]

# Remove invalid sentiment
hf_clean = hf_clean.dropna(subset=["sentiment"])

# Keep only negative and neutral
hf_clean = hf_clean[hf_clean["sentiment"].isin(["negative", "neutral"])].copy()

# Remove duplicates before sampling
hf_clean = hf_clean.drop_duplicates(subset=["review_text"])

print("HF clean sentiment distribution:")
print(hf_clean["sentiment"].value_counts())
print("\nNaN check:")
print(hf_clean.isna().sum())

HF clean sentiment distribution:
sentiment
negative    79819
neutral     39907
Name: count, dtype: int64

NaN check:
review_text    0
label          0
sentiment      0
dtype: int64


In [30]:
# Sample 30,000 negative and 30,000 neutral reviews

N_NEGATIVE = 30000
N_NEUTRAL = 30000

hf_negative_30k = hf_clean[hf_clean["sentiment"] == "negative"].sample(
    n=N_NEGATIVE,
    random_state=RANDOM_STATE
)

hf_neutral_30k = hf_clean[hf_clean["sentiment"] == "neutral"].sample(
    n=N_NEUTRAL,
    random_state=RANDOM_STATE
)

hf_30k_extra = pd.concat(
    [hf_negative_30k, hf_neutral_30k],
    ignore_index=True
)

print("Selected HF sample:")
print(hf_30k_extra["sentiment"].value_counts())

Selected HF sample:
sentiment
negative    30000
neutral     30000
Name: count, dtype: int64


In [31]:
# Normalize HF sample to match merged dataset schema

hf_30k_extra["rating"] = hf_30k_extra["label"] + 1
hf_30k_extra["product_name"] = "Unknown Product"
hf_30k_extra["raw_category"] = "Unknown Category"
hf_30k_extra["source"] = "hf_amazon_reviews_multi_en"
hf_30k_extra["label"] = hf_30k_extra["sentiment"].map(label2id).astype(int)

hf_30k_extra = hf_30k_extra[
    [
        "review_text",
        "rating",
        "product_name",
        "raw_category",
        "source",
        "sentiment",
        "label"
    ]
]

print("HF final sample:")
print(hf_30k_extra.shape)
print(hf_30k_extra.isna().sum())
print(hf_30k_extra["sentiment"].value_counts())

display(hf_30k_extra.head())

HF final sample:
(60000, 7)
review_text     0
rating          0
product_name    0
raw_category    0
source          0
sentiment       0
label           0
dtype: int64
sentiment
negative    30000
neutral     30000
Name: count, dtype: int64


,review_text,rating,product_name,raw_category,source,sentiment,label
0,Waste of money! Very flimsy. Broke after a wee...,1,Unknown Product,Unknown Category,hf_amazon_reviews_multi_en,negative,0
1,Does not spin freely on the drone. Drone will ...,1,Unknown Product,Unknown Category,hf_amazon_reviews_multi_en,negative,0
2,dislike clips are impossible to open you need ...,2,Unknown Product,Unknown Category,hf_amazon_reviews_multi_en,negative,0
3,I got this handbag for Christmas. I loved the ...,1,Unknown Product,Unknown Category,hf_amazon_reviews_multi_en,negative,0
4,A little larger than what I needed for phone a...,2,Unknown Product,Unknown Category,hf_amazon_reviews_multi_en,negative,0


In [32]:
# Add HF negative and neutral reviews to merged dataset

df_augmented = pd.concat(
    [df, hf_30k_extra],
    ignore_index=True
)

df_augmented = df_augmented.drop_duplicates(
    subset=["review_text", "rating", "sentiment"]
)

df_augmented = df_augmented.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("Original merged sentiment distribution:")
print(df["sentiment"].value_counts())

print("\nAugmented merged sentiment distribution:")
print(df_augmented["sentiment"].value_counts())

print("\nNaN check:")
print(df_augmented.isna().sum())

display(df_augmented.head())

Original merged sentiment distribution:
sentiment
positive    62545
negative    36361
neutral     27477
Name: count, dtype: int64

Augmented merged sentiment distribution:
sentiment
negative    65683
neutral     56630
positive    43041
Name: count, dtype: int64

NaN check:
review_text     0
rating          0
product_name    0
raw_category    0
source          0
sentiment       0
label           0
dtype: int64


,review_text,rating,product_name,raw_category,source,sentiment,label
0,"Not a great quality, look elsewhere. Cheaply m...",3.0,B07VCTRLDR,Sports_and_Outdoors,amazon_reviews_2023,neutral,1
1,DO NOT PURCHASE! We put the drain into the sin...,1.0,Unknown Product,Unknown Category,hf_amazon_reviews_multi_en,negative,0
2,I gave this as a gift and it did not work well...,1.0,Unknown Product,Unknown Category,hf_amazon_reviews_multi_en,negative,0
3,It offers many more options than I was looking...,5.0,"Fire Tablet, 7 Display, Wi - Fi, 8 GB - Includ...",Fire Tablets,primary,positive,2
4,Samsung has done it again. I love the kid feat...,5.0,"All - New Fire HD 8 Kids Edition Tablet, 8 HD ...",Fire Tablets,primary,positive,2


In [33]:
# Balance final dataset to 43,000 reviews per sentiment class

TARGET_PER_CLASS = 43000

balanced_df = (
    df_augmented
    .groupby("sentiment", group_keys=False)
    .apply(lambda x: x.sample(n=TARGET_PER_CLASS, random_state=RANDOM_STATE))
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print("Balanced dataset distribution:")
print(balanced_df["sentiment"].value_counts())

display(balanced_df.head())

Balanced dataset distribution:
sentiment
neutral     43000
negative    43000
positive    43000
Name: count, dtype: int64


,review_text,rating,product_name,raw_category,source,sentiment,label
0,I've been researching hot weather underwear fo...,3.0,Unknown Product,Unknown Category,hf_amazon_reviews_multi_en,neutral,1
1,I want to love this bra! Just needs a better b...,3.0,Unknown Product,Unknown Category,hf_amazon_reviews_multi_en,neutral,1
2,Care must be taken in cooking them after stuff...,1.0,B005QJMF2Q,Grocery_and_Gourmet_Food,amazon_reviews_2023,negative,0
3,Not worth it. I bought this pack because I nee...,1.0,B004AYCNR0,Video_Games,amazon_reviews_2023,negative,0
4,Side stove spiller covers. They cover the side...,3.0,B07WTXWC32,Appliances,amazon_reviews_2023,neutral,1


In [34]:
# Save balanced dataset for final EDA / modelling

BALANCED_OUTPUT_DIR = os.path.join(DATA_DIR, "merged")
os.makedirs(BALANCED_OUTPUT_DIR, exist_ok=True)

balanced_path = os.path.join(
    BALANCED_OUTPUT_DIR,
    "merged_dataset_clean.csv"
)

balanced_df.to_csv(balanced_path, index=False)

print("Balanced dataset saved in:")
print(balanced_path)

print("\nShape:")
print(balanced_df.shape)

print("\nSentiment distribution:")
print(balanced_df["sentiment"].value_counts())

Balanced dataset saved in:
data/merged/merged_dataset_clean.csv

Shape:
(129000, 7)

Sentiment distribution:
sentiment
neutral     43000
negative    43000
positive    43000
Name: count, dtype: int64


## 5. Data Cleaning and Sentiment Label Creation

This section cleans the merged dataset and creates the target variable.

The model does not directly learn from the star rating. The star rating is only used to create the supervised label:

- 1–2 stars → `negative`
- 3 stars → `neutral`
- 4–5 stars → `positive`

Rows with missing rating, missing review text, very short reviews, or duplicate review text are removed.

In [ ]:
# ============================================================
# Cleaning, label creation, and safe product display columns
# ============================================================

df["review_text"] = df["review_text"].apply(clean_text)
df["product_name"] = df["product_name"].apply(normalize_product_name)
df["raw_category"] = df["raw_category"].apply(normalize_category)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# Remove rows that cannot support supervised learning.
df = df.dropna(subset=["rating", "review_text"])
df = df[df["review_text"].str.len() >= 10]

# Create supervised target from rating.
df["sentiment"] = df["rating"].apply(map_rating_to_sentiment)
df = df.dropna(subset=["sentiment"])
df["label"] = df["sentiment"].map(label2id).astype(int)

# Product identifier quality fields.
df["product_id_type"] = df["product_name"].apply(product_id_type)
df["product_display_name"] = df.apply(
    lambda row: make_product_display_name(row["product_name"], row["raw_category"]),
    axis=1
)
df["product_display_short"] = df["product_display_name"].apply(lambda x: shorten_label(x, 75))

# Remove exact duplicated review/product/rating rows.
df = df.drop_duplicates(subset=["review_text", "rating", "product_name"])
df = df.reset_index(drop=True)

print("Clean merged dataset shape:", df.shape)
print("Product identifier quality:")
display(df["product_id_type"].value_counts().to_frame("count"))

display(df[[
    "review_text", "rating", "product_name", "product_id_type",
    "product_display_name", "raw_category", "source", "sentiment", "label"
]].tail())

Clean merged dataset shape: (163282, 10)
Product identifier quality:


,count
product_id_type,
asin,96913
named_product,59619
unknown,6750


,review_text,rating,product_name,product_id_type,product_display_name,raw_category,source,sentiment,label
163277,Original remote broken. Works really well.,5.0,B00ODIASKC,asin,Software product (B00ODIASKC),Software,amazon_reviews_2023,positive,2
163278,Exquisite!. Yo-Yo Ma with Emanuel Ax! None bet...,5.0,B0932HJ124,asin,CDs_and_Vinyl product (B0932HJ124),CDs_and_Vinyl,amazon_reviews_2023,positive,2
163279,Good value. I bought these rollers for quick t...,5.0,B0C28SB6D6,asin,Tools_and_Home_Improvement product (B0C28SB6D6),Tools_and_Home_Improvement,amazon_reviews_2023,positive,2
163280,Weak Output. The drop off in DB to a headphone...,2.0,B0002D03AW,asin,Musical_Instruments product (B0002D03AW),Musical_Instruments,amazon_reviews_2023,negative,0
163281,"In an emergency, the battery pack that works i...",5.0,B07S845G8W,asin,Cell_Phones_and_Accessories product (B07S845G8W),Cell_Phones_and_Accessories,amazon_reviews_2023,positive,2


In [ ]:
print(df["sentiment"].value_counts())

print(df["sentiment"].value_counts(normalize=True)*100)

sentiment
positive    139818
negative     13047
neutral      10417
Name: count, dtype: int64
sentiment
positive    85.629769
negative     7.990470
neutral      6.379760
Name: proportion, dtype: float64


**6. EDA**

**6. BALANCING**

**7. BASELINE MODEL**

In [ ]:
train_df, test_df = train_test_split(
    balanced_df,
    test_size=0.2,
    stratify=balanced_df["sentiment"],
    random_state=42
)

print("Train set shape:", train_df.shape)
print("Test set shape:", test_df.shape)

In [ ]:
# ============================================================
# Save a copy of the full balanced dataset for clustering
# ============================================================
# We save balanced_df NOW, before any model training.
# The clustering step (Section 10) will use this copy directly —
# it does not need train/test split, it works on the full dataset.
# This also ensures clustering is never accidentally affected
# by any transformation we apply to train_df / test_df later.

df_for_clustering = balanced_df.copy()

print("Dataset saved for clustering:")
print(df_for_clustering.shape)
print(df_for_clustering["sentiment"].value_counts())

In [ ]:
# ============================================================
# Step 1: Zero-shot baseline — DistilBERT WITHOUT fine-tuning
# ============================================================
# Before we fine-tune anything, we test the off-the-shelf DistilBERT
# sentiment model directly on our data.
#
# WHY: Most pre-trained sentiment models on HuggingFace are BINARY
# (positive / negative only). They have no concept of "neutral".
# This cell demonstrates that problem concretely:
# → recall for "neutral" will be 0% because the model can never
#   predict a class it wasn't trained on.
# This is the key justification for why fine-tuning is necessary.

zero_shot_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=MAX_TOKEN_LENGTH,
    device=0 if device == "cuda" else -1
)

# Evaluate on a sample of 500 rows (faster, representative)
sample_eval = test_df.sample(500, random_state=42).copy()

zero_shot_preds = zero_shot_pipe(
    sample_eval["review_text"].tolist(),
    batch_size=32
)

# Map binary output (POSITIVE/NEGATIVE) to our 3-class space
def map_binary_to_3class(label):
    """
    A binary model outputs only POSITIVE or NEGATIVE.
    It structurally cannot predict NEUTRAL — recall for neutral = 0%.
    """
    return "positive" if label == "POSITIVE" else "negative"

sample_eval["zero_shot_pred"] = [
    map_binary_to_3class(p["label"]) for p in zero_shot_preds
]

print("=== ZERO-SHOT BASELINE (no fine-tuning) ===")
print("Model: distilbert-base-uncased-finetuned-sst-2-english")
print()
print(classification_report(
    sample_eval["sentiment"],
    sample_eval["zero_shot_pred"],
    labels=["negative", "neutral", "positive"],
    zero_division=0
))
print()
print("[NOTE] Recall for 'neutral' = 0%: the off-the-shelf binary model")
print("has no neutral class to predict. Fine-tuning is required.")

In [ ]:
!pip uninstall -y torchvision
import sys
# Force remove torchvision from the current session's memory
for mod in list(sys.modules.keys()):
    if "torchvision" in mod:
        del sys.modules[mod]
print("torchvision removed from session")

In [ ]:
try:
    import torchvision
    print("WARNING: torchvision still importable")
except ImportError:
    print("OK: torchvision not available")

In [ ]:
# Fix labels before fine-tuning
# 1. Drop rows with NaN labels
train_df = train_df.dropna(subset=["label"])
test_df = test_df.dropna(subset=["label"])

# 2. Convert labels to int
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

# Verify
print("Train label dtype:", train_df["label"].dtype)
print("Train unique labels:", sorted(train_df["label"].unique()))
print("Train label distribution:")
print(train_df["label"].value_counts().sort_index())
print()
print("Test label distribution:")
print(test_df["label"].value_counts().sort_index())
print()
print("Any NaN remaining?", train_df["label"].isna().any())

In [ ]:
# ============================================================
# Step 2: Fine-tune DistilBERT for 3-class sentiment classification
# ============================================================
# We take DistilBERT (pre-trained on general language understanding)
# and adapt it to our specific task: predict negative / neutral / positive.
#
# What changes during fine-tuning:
# - The last classification layer is replaced: 2 outputs → 3 outputs
# - All weights are slightly adjusted on our balanced dataset
#   (103k reviews, 43k per class) using a small learning rate (2e-5)
#   to avoid "forgetting" what the model already knows about language
#   (catastrophic forgetting)
#
# Why DistilBERT as baseline?
# - 40% lighter than BERT, 97% of BERT's accuracy
# - Fast to train (~15 min on Colab GPU)
# - Strong starting point to beat with RoBERTa in Section 8

if RUN_TRAINING:
    print("Starting DistilBERT fine-tuning...")
    print(f"Train size: {len(train_df)} | Test size: {len(test_df)}")
    print(f"Epochs: {NUM_EPOCHS} | Batch size: {BATCH_SIZE} | LR: {LEARNING_RATE}")
    print()

    baseline_result = train_transformer_classifier(
        train_df=train_df,
        test_df=test_df,
        model_name=BASELINE_MODEL_NAME,       # "distilbert-base-uncased"
        output_subdir="baseline_distilbert",
        epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE
    )

    print()
    print("=== BASELINE RESULTS (DistilBERT fine-tuned) ===")
    print(f"Accuracy   : {baseline_result['accuracy']:.4f}")
    print(f"Macro F1   : {baseline_result['macro_f1']:.4f}")
    print(f"Weighted F1: {baseline_result['weighted_f1']:.4f}")
    print()
    print("[INFO] Model saved to:", baseline_result['output_path'])

else:
    print("RUN_TRAINING = False — skipping DistilBERT fine-tuning.")
    baseline_result = None

**8. COMPARISON MODEL**

In [ ]:
!pip install -U datasets
!pip uninstall -y torchvision

In [ ]:
# ============================================================
# PREPARE LABELS FOR TRANSFORMER MODELS
# ============================================================

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

train_df = train_df.copy()
test_df = test_df.copy()

train_df["label"] = train_df["sentiment"].map(label2id)
test_df["label"] = test_df["sentiment"].map(label2id)

train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain distribution:")
print(train_df["sentiment"].value_counts())

print("\nTest distribution:")
print(test_df["sentiment"].value_counts())

In [ ]:
# ============================================================
# 8. COMPARISON MODEL - RoBERTa
# ============================================================

roberta_results = train_transformer_classifier(
    train_df=train_df,
    test_df=test_df,
    model_name="roberta-base",
    output_subdir="roberta_sentiment_model",
    epochs=2,
    batch_size=8,
    learning_rate=2e-5
)

print("RoBERTa Accuracy:", roberta_results["accuracy"])
print("RoBERTa Macro F1:", roberta_results["macro_f1"])
print("RoBERTa Weighted F1:", roberta_results["weighted_f1"])

**9. EVALUATION**

**10. CLUSTERING**

**11. SUMMARIZATION**

**12. APP**

**13. REPORT + DERIVABLES**